# Comparing John and Julia's embeddings 
This program takes a sample of SGA 2025 galaxies and finds the difference between embedding sources

In [1]:
# ============================================================
# OFFICIAL SGA EMBEDDING TEST SET
# SGAML KERNEL
# ============================================================

import h5py
import numpy as np
from pathlib import Path

from SGA.SGA import read_sga_sample
from SGA.ssl import load_ssl_embeddings


# ============================================================
# Configuration
# ============================================================

SSL_DIR = Path(
    "/global/cfs/cdirs/desicollab/users/ioannis/SGA/2025/ssl"
)

REGION = "dr11-south"

CHUNK_PATH = (
    "/global/cfs/cdirs/desicollab/users/ioannis"
    "/SGA/2025/ssl/ssl-cutouts-dr11-south-chunk0007.hdf5"
)

N_TEST = 100

OUTPUT_PATH = Path(
    "/pscratch/sd/q/qshimp/SGA2025-data/"
    "embedding_comparison_100.npz"
)


# ============================================================
# Read the 100 test galaxies from chunk 0007
# ============================================================

with h5py.File(CHUNK_PATH, "r") as f:

    n_galaxies = f["sgaid"].shape[0]

    chunk_ids = f["sgaid"][:N_TEST]
    chunk_ra = f["ra"][:N_TEST]
    chunk_dec = f["dec"][:N_TEST]

print(f"Galaxies in chunk 0007: {n_galaxies:,}")
print(f"Test galaxies:          {len(chunk_ids)}")


# ============================================================
# Load official SGA embeddings
# ============================================================

catalog, _ = read_sga_sample(
    region=REGION,
    no_groups=True
)

existing_gals = load_ssl_embeddings(
    REGION,
    SSL_DIR,
    catalog
)

existing_ids = np.asarray(
    existing_gals["SGAID"]
)

existing_embeddings = np.asarray(
    existing_gals["embeddings"]
)

print(
    f"Official SGA galaxies:   {len(existing_ids):,}"
)

print(
    f"Official embedding shape: "
    f"{existing_embeddings.shape}"
)


# ============================================================
# Match chunk galaxies to official embeddings
# ============================================================

id_lookup = {
    int(sgaid): i
    for i, sgaid in enumerate(existing_ids)
}


missing_ids = [
    int(sgaid)
    for sgaid in chunk_ids
    if int(sgaid) not in id_lookup
]

if missing_ids:

    raise ValueError(
        f"{len(missing_ids)} test galaxies were not "
        f"found in the official SGA embeddings."
    )


official_indices = np.array([
    id_lookup[int(sgaid)]
    for sgaid in chunk_ids
])


official_test_embeddings = (
    existing_embeddings[official_indices]
)


official_test_ids = (
    existing_ids[official_indices]
)


# ============================================================
# Verify alignment
# ============================================================

if not np.array_equal(
    chunk_ids,
    official_test_ids
):

    raise ValueError(
        "SGAID ordering does not match!"
    )

print("SGAID ordering verified.")


# ============================================================
# Save test data
# ============================================================

np.savez(
    OUTPUT_PATH,
    sgaid=chunk_ids,
    ra=chunk_ra,
    dec=chunk_dec,
    official_embeddings=official_test_embeddings,
)

print()
print(f"Saved test data to:")
print(OUTPUT_PATH)

print()
print("Saved:")
print(f"  SGAIDs:              {chunk_ids.shape}")
print(f"  Official embeddings: {official_test_embeddings.shape}")

Galaxies in chunk 0007: 45,450
Test galaxies:          100
INFO:SGA.py:363:_read_catalog: Read 395,435/395,435 GROUP_PRIMARY objects from /dvs_ro/cfs/cdirs/cosmo/work/legacysurvey/sga/2025/sample/SGA2025-beta-v1.6-dr11-south.fits
INFO:SGA.py:370:_read_catalog: Selecting 395,435/395,435 objects in region=dr11-south
INFO:ssl.py:323:load_ssl_embeddings: load_ssl_embeddings: 363,607 objects from /global/cfs/cdirs/desicollab/users/ioannis/SGA/2025/ssl/ssl-embeddings-dr11-south.hdf5
Official SGA galaxies:   363,607
Official embedding shape: (363607, 2048)
SGAID ordering verified.

Saved test data to:
/pscratch/sd/q/qshimp/SGA2025-data/embedding_comparison_100.npz

Saved:
  SGAIDs:              (100,)
  Official embeddings: (100, 2048)
